In [4]:
import gspread

gc = gspread.service_account(filename="service_account.json")
ws = gc.open_by_key("1YMMSChT6Khu_PFP5ekOT8qCqVgce9XsflTrcBo3a5ZY").sheet1

ws.append_row([
    "117",
    "Test raw lyrics",
    "",
    "https://example.com"
])

print("Success!")

Success!


In [7]:
import time
import requests
import gspread
import re
from bs4 import BeautifulSoup

# =========================
# SETTINGS
# =========================
SHEET_KEY = "1YMMSChT6Khu_PFP5ekOT8qCqVgce9XsflTrcBo3a5ZY"
WORKSHEET_NAME = "Sheet1"
START_NUM = 329
END_NUM = 750
DELAY_SECONDS = 0.5

# =========================
# GOOGLE SHEETS HELPERS
# =========================
def get_worksheet():
    gc = gspread.service_account("service_account.json")
    sh = gc.open_by_key(SHEET_KEY)
    ws = sh.worksheet(WORKSHEET_NAME)
    return ws

def ensure_headers(ws):
    headers = ws.row_values(1)
    expected = ["UMH Number", "Title", "Lyrics (Raw)", "Source", "Status"]
    if headers != expected:
        ws.update("A1:E1", [expected])

def existing_umh_numbers(ws):
    values = ws.col_values(1)  # column A
    nums = set()

    for value in values[1:]:  # skip header
        value = value.strip()
        if value.isdigit():
            nums.add(int(value))

    return nums

# =========================
# SCRAPING HELPERS
# =========================
def build_url(num: int) -> str:
    return f"https://www.hymnsite.com/lyrics/umh{num:03}.txt"

def fetch_title(num):
    url = f"https://www.hymnsite.com/lyrics/umh{num:03}.sht"
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers, timeout=20)
    if r.status_code != 200:
        return ""

    soup = BeautifulSoup(r.text, "html.parser")
    title = soup.title.text.strip()
    title = re.sub(r"\s*-\s*HymnSite\.com.*$", "", title).strip()
    return title


def fetch_lyrics(url: str) -> str | None:
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    try:
        response = requests.get(url, headers=headers, timeout=20)
    except requests.RequestException:
        return None

    if response.status_code != 200:
        return None

    text = response.text.strip()

    # Skip empty or obviously invalid pages
    if not text:
        return None

    return text



def append_hymn(ws, num: int, title: str, lyrics: str, source: str):
    ws.append_row([
        str(num),
        title,
        lyrics,
        "",
        source,
        "Imported"
    ])

# =========================
# MAIN
# =========================
def main():
    ws = get_worksheet()
    ensure_headers(ws)
    existing = existing_umh_numbers(ws)

    added = 0
    skipped_existing = 0
    missing = 0
    failed = 0

    for num in range(START_NUM, END_NUM + 1):
        if num in existing:
            print(f"UMH {num:03}: already exists, skipped")
            skipped_existing += 1
            continue

        url = build_url(num)
        title = fetch_title(num)
        lyrics = fetch_lyrics(url)

        if lyrics is None:
            print(f"UMH {num:03}: missing or unavailable")
            missing += 1
            time.sleep(DELAY_SECONDS)
            continue

        try:
            append_hymn(ws, num, title, lyrics, url)
            print(f"UMH {num:03}: added")
            added += 1
        except Exception as e:
            print(f"UMH {num:03}: failed to write to sheet -> {e}")
            failed += 1

        time.sleep(DELAY_SECONDS)

    print("\nImport complete.")
    print(f"Added: {added}")
    print(f"Already existed: {skipped_existing}")
    print(f"Missing/unavailable: {missing}")
    print(f"Failed: {failed}")

if __name__ == "__main__":
    main()

UMH 329: missing or unavailable
UMH 330: missing or unavailable
UMH 331: missing or unavailable
UMH 332: added
UMH 333: missing or unavailable
UMH 334: missing or unavailable
UMH 335: missing or unavailable
UMH 336: missing or unavailable
UMH 337: added
UMH 338: added
UMH 339: added
UMH 340: missing or unavailable
UMH 341: added
UMH 342: missing or unavailable
UMH 343: missing or unavailable
UMH 344: missing or unavailable
UMH 345: missing or unavailable
UMH 346: missing or unavailable
UMH 347: missing or unavailable
UMH 348: added
UMH 349: missing or unavailable
UMH 350: missing or unavailable
UMH 351: added
UMH 352: missing or unavailable
UMH 353: missing or unavailable
UMH 354: added
UMH 355: added
UMH 356: added
UMH 357: added
UMH 358: added
UMH 359: added
UMH 360: missing or unavailable
UMH 361: added
UMH 362: added
UMH 363: added
UMH 364: missing or unavailable
UMH 365: added
UMH 366: missing or unavailable
UMH 367: missing or unavailable
UMH 368: added
UMH 369: added
UMH 370: mi

KeyboardInterrupt: 

In [2]:
from pptx import Presentation

prs = Presentation("/Users/weixiaojing/Desktop/WSCS PPT Layout (1).pptx")

for layout in prs.slide_layouts:
    print(layout.name)


=== Slide 0 ===
Shape 0: '{{TITLE}}'
Shape 1: '{{LYRICS}}'
Shape 2: '[TEMPLATE_FIRST]'

=== Slide 1 ===
Shape 0: '[TEMPLATE_REST]'
Shape 1: '{{LYRICS}}'

--- Summary ---
FIRST TEMPLATE SLIDE: 0
REST TEMPLATE SLIDE: 1
✅ First template has {{TITLE}}
✅ First template has {{LYRICS}}
✅ Rest template has {{LYRICS}}
